# SHBT261 TextVQA Full Prompt-Engineering Pipeline

This notebook runs the full 5,000-example TextVQA validation experiment using Qwen2.5-VL-3B-Instruct and four prompt conditions: plain, concise, OCR-aware, and strict OCR-aware. It saves all predictions, metrics, report tables, plots, and qualitative examples inside this uploaded `textvqa_final` folder.


In [1]:
# Install dependencies. Runtime restart is usually not needed after this cell.
import sys, subprocess
packages = [
    'transformers>=4.51.0', 'accelerate>=0.34.0', 'bitsandbytes>=0.43.3',
    'datasets>=2.20.0', 'qwen-vl-utils[decord]', 'nltk', 'matplotlib',
    'seaborn', 'pandas', 'tabulate',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U', *packages])


0

In [2]:
from google.colab import drive
drive.mount('/content/drive')

# Upload the folder directly to: MyDrive/textvqa_final
PROJECT_DIR = '/content/drive/MyDrive/textvqa_final'
CODE_DIR = PROJECT_DIR

import os, sys
os.makedirs(PROJECT_DIR, exist_ok=True)
sys.path.insert(0, CODE_DIR)

print('Project dir:', PROJECT_DIR)
print('Code dir:', CODE_DIR)
print('Files:', sorted(os.listdir(CODE_DIR)))


Mounted at /content/drive
Project dir: /content/drive/MyDrive/textvqa_final
Code dir: /content/drive/MyDrive/textvqa_final
Files: ['README.md', '__pycache__', 'colab_prompt_engineering_final.ipynb', 'textvqa_pipeline.py']


In [3]:
import importlib
import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')

import textvqa_pipeline
importlib.reload(textvqa_pipeline)
from textvqa_pipeline import *

base_config = RunConfig(
    project_dir=PROJECT_DIR,
    max_new_tokens=8,
    inference_batch_size=4,
)

set_seed(base_config.seed)
preflight()
ensure_dirs(PROJECT_DIR)


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
VRAM: 79.3 GB


{'root': PosixPath('/content/drive/MyDrive/textvqa_final'),
 'results': PosixPath('/content/drive/MyDrive/textvqa_final/results'),
 'figures': PosixPath('/content/drive/MyDrive/textvqa_final/figures')}

## Full Validation Prompt Ablation

This cell runs all four prompt styles on the full 5,000-example validation split. It writes partial predictions every 250 samples and final predictions/metrics for each prompt to `textvqa_final/results/`.


In [4]:
PROMPT_STYLES = ['plain', 'concise', 'ocr', 'strict_ocr']
VAL_N = None  # None means the full 5,000-example validation split

model, processor = get_model_and_processor(base_config)
val_ds = load_textvqa_split('validation', max_samples=VAL_N)

prompt_runs = {}
for style in PROMPT_STYLES:
    prefix = f'prompt_{style}_valfull'
    results, metrics = run_inference(
        model, processor, val_ds, base_config,
        output_prefix=prefix,
        prompt_style=style,
        max_samples=VAL_N,
    )
    prompt_runs[style] = {'prefix': prefix, 'metrics': metrics}

prompt_runs


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

inference:prompt_ocr_valfull:   0%|          | 0/1250 [00:00<?, ?it/s]


prompt_ocr_valfull
Samples: 5000
            accuracy:  68.95
         exact_match:  73.46
                  f1:  80.14
                bleu:  78.56
              meteor:  53.11
             rouge_l:  80.12
 substring_precision:  79.02
    substring_recall:  80.02

Per-category accuracy:
        what_general:  67.39  n=1566
              number:  68.58  n=975
               brand:  80.62  n=609
        reading_text:  64.89  n=600
               other:  60.31  n=409
              object:  67.20  n=313
              person:  76.68  n=233
               place:  74.41  n=198
               which:  63.93  n=61
               color:  59.26  n=36


inference:prompt_strict_ocr_valfull:   0%|          | 0/1250 [00:00<?, ?it/s]


prompt_strict_ocr_valfull
Samples: 5000
            accuracy:  73.81
         exact_match:  77.98
                  f1:  83.59
                bleu:  82.26
              meteor:  54.68
             rouge_l:  83.55
 substring_precision:  82.78
    substring_recall:  83.40

Per-category accuracy:
        what_general:  71.69  n=1566
              number:  72.00  n=975
               brand:  82.70  n=609
        reading_text:  67.94  n=600
               other:  77.91  n=409
              object:  70.50  n=313
              person:  82.26  n=233
               place:  79.29  n=198
               which:  65.57  n=61
               color:  73.15  n=36


{'plain': {'prefix': 'prompt_plain_valfull',
  'metrics': {'num_samples': 5000,
   'accuracy': 0.0002666666666666666,
   'exact_match': 0.0008,
   'f1': 0.12967261411613712,
   'bleu': 0.047043766827970686,
   'meteor': 0.13814542576001704,
   'rouge_l': 0.12755283056635358,
   'substring_precision': 0.001,
   'substring_recall': 0.2536,
   'per_category': {'brand': {'count': 609, 'accuracy': 0.0005473453749315818},
    'color': {'count': 36, 'accuracy': 0.0},
    'number': {'count': 975, 'accuracy': 0.0003418803418803419},
    'object': {'count': 313, 'accuracy': 0.0},
    'other': {'count': 409, 'accuracy': 0.0},
    'person': {'count': 233, 'accuracy': 0.0},
    'place': {'count': 198, 'accuracy': 0.0},
    'reading_text': {'count': 600, 'accuracy': 0.0005555555555555556},
    'what_general': {'count': 1566, 'accuracy': 0.00021285653469561513},
    'which': {'count': 61, 'accuracy': 0.0}}}},
 'concise': {'prefix': 'prompt_concise_valfull',
  'metrics': {'num_samples': 5000,
   'accu

## Report Assets

This cell creates the report-ready outputs: summary tables, LaTeX tables, category heatmaps, metric comparison plots, error analysis, OCR-count and answer-length diagnostics, transition plots, and qualitative examples.


In [5]:
from pathlib import Path

results_dir = Path(PROJECT_DIR) / 'results'
prompt_prefixes = [prompt_runs[style]['prefix'] for style in PROMPT_STYLES]

for prefix in prompt_prefixes:
    analyze_predictions(results_dir / f'{prefix}_predictions.json', prefix, PROJECT_DIR)

summary = summarize_prompt_runs(prompt_prefixes, PROJECT_DIR, output_name='full_prompt_ablation_summary')
engineered_styles = ['concise', 'ocr', 'strict_ocr']
best_engineered_style = max(engineered_styles, key=lambda s: prompt_runs[s]['metrics']['accuracy'])
best_engineered_prefix = prompt_runs[best_engineered_style]['prefix']

compare_prediction_files(
    results_dir / 'prompt_plain_valfull_predictions.json',
    results_dir / f'{best_engineered_prefix}_predictions.json',
    'plain_baseline', f'best_engineered_{best_engineered_style}', PROJECT_DIR,
)
compare_prediction_files(
    results_dir / 'prompt_concise_valfull_predictions.json',
    results_dir / f'{best_engineered_prefix}_predictions.json',
    'concise_baseline', f'best_engineered_{best_engineered_style}', PROJECT_DIR,
)

make_plots([results_dir / f'{prefix}_metrics.json' for prefix in prompt_prefixes], PROJECT_DIR)
report_assets = create_report_assets(
    prompt_prefixes,
    PROJECT_DIR,
    baseline_prefix='prompt_concise_valfull',
    best_prefix=best_engineered_prefix,
    output_name='full_prompt_report_assets',
)

print('Best engineered prompt:', best_engineered_style)
print('Summary:', results_dir / 'full_prompt_ablation_summary.csv')
print('Report tables:', results_dir / 'report_tables')
print('Report examples:', results_dir / 'report_examples')
print('Figures:', Path(PROJECT_DIR) / 'figures')
report_assets


Best engineered prompt: strict_ocr
Summary: /content/drive/MyDrive/textvqa_final/results/full_prompt_ablation_summary.csv
Report tables: /content/drive/MyDrive/textvqa_final/results/report_tables
Report examples: /content/drive/MyDrive/textvqa_final/results/report_examples
Figures: /content/drive/MyDrive/textvqa_final/figures


{'metrics_table_csv': '/content/drive/MyDrive/textvqa_final/results/report_tables/full_prompt_report_assets_metrics_table.csv',
 'metrics_table_markdown': '/content/drive/MyDrive/textvqa_final/results/report_tables/full_prompt_report_assets_metrics_table.md',
 'metrics_table_latex': '/content/drive/MyDrive/textvqa_final/results/report_tables/full_prompt_report_assets_metrics_table.tex',
 'per_sample_diagnostics_csv': '/content/drive/MyDrive/textvqa_final/results/report_tables/full_prompt_report_assets_per_sample_diagnostics.csv',
 'category_accuracy_csv': '/content/drive/MyDrive/textvqa_final/results/report_tables/full_prompt_report_assets_category_accuracy.csv',
 'category_accuracy_markdown': '/content/drive/MyDrive/textvqa_final/results/report_tables/full_prompt_report_assets_category_accuracy.md',
 'metric_comparison_figure': '/content/drive/MyDrive/textvqa_final/figures/full_prompt_report_assets_metric_comparison.png',
 'qualitative_examples_markdown': '/content/drive/MyDrive/textv

In [7]:
import importlib, sys
from pathlib import Path
from collections import Counter

PROJECT_DIR = "/content/drive/MyDrive/textvqa_final"
CODE_DIR = PROJECT_DIR

if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

import textvqa_pipeline
importlib.reload(textvqa_pipeline)
from textvqa_pipeline import *

prompt_prefixes = [
    "prompt_plain_valfull",
    "prompt_concise_valfull",
    "prompt_ocr_valfull",
    "prompt_strict_ocr_valfull",
]

# Fast sanity check: estimates how many examples need the LLM judge.
for prefix in prompt_prefixes:
    rows = load_json(Path(PROJECT_DIR) / "results" / f"{prefix}_predictions.json")
    counts = Counter()
    for row in rows:
        if textvqa_accuracy(row.get("prediction", ""), row.get("ground_truths", [])) > 0:
            counts["AUTO_EXACT_MATCH"] += 1
        else:
            score, reason = strict_judge_prefilter(
                row.get("question", ""),
                row.get("prediction", ""),
                row.get("ground_truths", []),
            )
            counts["LLM_CALL" if score is None else reason] += 1
    print(prefix, dict(counts))


prompt_plain_valfull {'AUTO_QUESTION_RESTATEMENT': 2061, 'LLM_CALL': 2694, 'AUTO_INCOMPLETE_FRAGMENT': 241, 'AUTO_EXACT_MATCH': 4}
prompt_concise_valfull {'AUTO_EXACT_MATCH': 3674, 'LLM_CALL': 1311, 'AUTO_QUESTION_RESTATEMENT': 9, 'AUTO_EMPTY_PREDICTION': 3, 'AUTO_NO_CONTENT': 2, 'AUTO_INCOMPLETE_FRAGMENT': 1}
prompt_ocr_valfull {'AUTO_EXACT_MATCH': 3673, 'LLM_CALL': 1312, 'AUTO_QUESTION_RESTATEMENT': 8, 'AUTO_NO_CONTENT': 3, 'AUTO_EMPTY_PREDICTION': 2, 'AUTO_INCOMPLETE_FRAGMENT': 2}
prompt_strict_ocr_valfull {'AUTO_EXACT_MATCH': 3899, 'LLM_CALL': 1087, 'AUTO_QUESTION_RESTATEMENT': 8, 'AUTO_EMPTY_PREDICTION': 3, 'AUTO_INCOMPLETE_FRAGMENT': 1, 'AUTO_NO_CONTENT': 2}


In [8]:
strict_judge_summary = judge_predictions_with_llm(
    prefixes=prompt_prefixes,
    project_dir=PROJECT_DIR,
    judge_model_name="Qwen/Qwen2.5-7B-Instruct",
    batch_size=16,
    output_name="llm_judge_similarity_strict7b",
    use_strict_prefilter=True,
    resume=True,
    checkpoint_every_batches=25,
)

strict_judge_summary


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

judge:prompt_plain_valfull:   0%|          | 0/169 [00:00<?, ?it/s]

judge:prompt_concise_valfull:   0%|          | 0/82 [00:00<?, ?it/s]

judge:prompt_ocr_valfull:   0%|          | 0/82 [00:00<?, ?it/s]

judge:prompt_strict_ocr_valfull:   0%|          | 0/68 [00:00<?, ?it/s]

{'judge_model': 'Qwen/Qwen2.5-7B-Instruct',
 'runs': [{'run': 'prompt_plain_valfull',
   'num_samples': 5000,
   'llm_judge_similarity': 0.207,
   'llm_judge_similarity_percent': 20.7,
   'judge_model': 'Qwen/Qwen2.5-7B-Instruct',
   'llm_calls': 2694,
   'auto_exact_or_empty': 4,
   'auto_question_restatement': 2061,
   'auto_incomplete_fragment': 241,
   'auto_no_content': 0,
   'strict_prefilter': True},
  {'run': 'prompt_concise_valfull',
   'num_samples': 5000,
   'llm_judge_similarity': 0.819,
   'llm_judge_similarity_percent': 81.89999999999999,
   'judge_model': 'Qwen/Qwen2.5-7B-Instruct',
   'llm_calls': 1311,
   'auto_exact_or_empty': 3677,
   'auto_question_restatement': 9,
   'auto_incomplete_fragment': 1,
   'auto_no_content': 2,
   'strict_prefilter': True},
  {'run': 'prompt_ocr_valfull',
   'num_samples': 5000,
   'llm_judge_similarity': 0.8138,
   'llm_judge_similarity_percent': 81.38,
   'judge_model': 'Qwen/Qwen2.5-7B-Instruct',
   'llm_calls': 1312,
   'auto_exact_o